# Predicción de Enfermedades Cardíacas

Proyecto de clasificación listo para **Google Colab**.

**Pregunta:** ¿podemos predecir la presencia de enfermedad cardíaca a partir de indicadores clínicos?

**Target:** `target` (1 = enfermedad, 0 = no)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico Heart Disease (UCI / Kaggle).

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv"
df = pd.read_csv(url)
print(df.shape)
print(df["target"].value_counts())
df.head()

## 2. EDA

In [ ]:
print("Tasa de enfermedad:", df["target"].mean().round(3))
df.describe().T

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target")
plt.xticks([0, 1], ["No", "Sí"])
plt.title("Distribución de enfermedad cardíaca")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(data=df, x="age", hue="target", kde=True, ax=axes[0])
sns.histplot(data=df, x="trestbps", hue="target", kde=True, ax=axes[1])
sns.histplot(data=df, x="chol", hue="target", kde=True, ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlación")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="cp", hue="target", ax=axes[0])
axes[0].set_title("Tipo de dolor de pecho vs target")
sns.countplot(data=df, x="sex", hue="target", ax=axes[1])
axes[1].set_title("Sexo vs target")
plt.tight_layout()
plt.show()

### Hallazgos
- Dataset relativamente balanceado.
- `cp` (tipo de dolor), `thalach` y `exang` correlacionan con el target.
- Edad y sexo también influyen.

## 3. Preparar datos

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Random Forest"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No", "Sí"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No", "Sí"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Matriz de confusión")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

## 6. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(7, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importancia — Random Forest")
plt.tight_layout()
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values[1], X_train_scaled, feature_names=X.columns, show=False)
plt.title("SHAP Summary — enfermedad cardíaca")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values[1], X_train_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Variables como `cp`, `thalach`, `oldpeak` y `exang` suelen ser las más importantes.
- ROC-AUC suele estar en **0.85–0.92** en test.
- Dataset pequeño pero clásico para clasificación médica.

**Rango esperado:** Accuracy ~0.80–0.88 | ROC-AUC ~0.87–0.93